In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
print('here')

here


In [3]:
import sqlite3
from datetime import datetime
from pathlib import Path
import pandas as pd

import arcgis
from arcgis.gis import GIS
import sys

# 1. Bepaal het absolute pad van de bovenliggende map (de project root)
project_root = str(Path.cwd().parent)

# 2. Voeg de project root toe aan sys.path, zodat Python de 'src' map kan vinden
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.utils as utils
import json

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# Create and publish surveys

<div class="alert alert-block alert-info">This notebook uses the ArcGIS API for Python. For more information, see the <a href="https://developers.arcgis.com/python/">ArcGIS API for Python documentation and guides</a>.</div>

 The `create()` method is useful for batch creating blank surveys for which you can reassign ownership to another colleague to complete the design and publish. If you already have an XLSForm, you can use the `create()` and `publish()` methods to automate creating and publishing surveys.

 This sample notebook demonstrates how you can automate creating and publishing surveys using ArcGIS API for Python.

In [4]:
# Get credentials from key vault into environment
# Define your local specific paths 
LOCAL_DB = "G:\Mijn Drive\keepass_db.kdbx"
ENTRY_TITLE = "AGOL"

# Run the function
AGOL_USER, AGOL_PASS = utils.load_keepass_credentials(db_path=LOCAL_DB, entry_name=ENTRY_TITLE)

✅ Success: Credentials for 'AGOL' loaded into environment variables!


In [12]:
agol_url = "https://gisservices.inbo.be/portal"
gis = GIS(agol_url, AGOL_USER, AGOL_PASS)

In [13]:
from pathlib import Path
import re

In [14]:
xlsform_path = r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab1-4.xlsx"  # Pad naar het XLSForm-bestand
target_folder = "Survey-LSVI App Test Auto"

## Update existing survey

In [18]:
survey_manager = arcgis.apps.survey123.SurveyManager(gis)
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab1-4.xlsx")
survey_name = "LSVI App Test Auto" 
surveys = survey_manager.surveys
    
# Safely find matches using the .get() method
matched_surveys = [s for s in surveys if s.properties.get('title') == survey_name]

if matched_surveys:
    # Return the ID of the first matching survey found
    print(matched_surveys[0].properties.get('id'))

# If nothing matches, gather available titles to provide a helpful error message
available_titles = [s.properties.get('title') for s in surveys if s.properties.get('title')]
print(f"Survey '{survey_name}' not found in the connected portal.\n")
print(f"Available surveys: {available_titles}")

Connection to 172.31.1.204:9876 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.
(Error Code: 500)


TypeError: unsupported operand type(s) for +: 'NoneType' and 'NoneType'

In [15]:
# Instatiate surveymanager object
survey_manager = arcgis.apps.survey123.SurveyManager(gis)
FIELD_MAP_ID = '64c1f0bd02344d5ebf41c3dd320615bc'

# Path to XLSForm file
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab1-4.xlsx")
survey_name = xlsform_path.stem 

# Fetch old survey ID by name
old_survey_id = utils.get_survey_id_by_name(survey_manager, survey_name)
print(f"Bestaande survey ({old_survey_id}) aan het updaten...")
    
# Haal de bestaande survey ops
mijn_survey = survey_manager.get(old_survey_id)

Connection to 172.31.1.204:9876 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.
(Error Code: 500)


TypeError: unsupported operand type(s) for +: 'NoneType' and 'NoneType'

In [ ]:


# Overschrijf het formulier met het nieuwe Excel-bestand
mijn_survey.publish(
    xlsform=str(xlsform_path),
    schema_changes=True,  # Sta schemawijzigingen toe
)
print("Survey succesvol overschreven!")

# Fetch object ID of new survey
new_survey_id = utils.get_survey_id_by_name(survey_manager, survey_name)
print(f"Nieuw survey object aangemaakt: {new_survey_id}")

# Get the web map's internal data configuration
map_item = gis.content.get(FIELD_MAP_ID)
map_data = map_item.get_data()

# Convert configuration to a string to easily replace the old ID globally
map_data_str = json.dumps(map_data)

# Scenario A: The survey ID did not change (e.g., standard overwrite publish)
if old_survey_id == new_survey_id:
    print("ℹ️ The survey ID did not change. No web map update is required.")

# Scenario B: The survey ID changed (e.g., wipe and recreate workflow)
elif old_survey_id in map_data_str:
    # Safely swap out ONLY the old target ID with the new one
    updated_map_data_str = map_data_str.replace(old_survey_id, new_survey_id)
    updated_map_data = json.loads(updated_map_data_str)
    
    # Push the updated data config back to the Web Map item in AGOL
    map_item.update(data=updated_map_data)
    print(f"Success! Updated the pop-up link in the Web Map from {old_survey_id} to {new_survey_id}.")

# Scenario C: Safety net in case the survey ID isn't found in the popup at all
else:
    print(
        f"⚠️ Warning: The old survey ID ({old_survey_id}) for '{survey_name}' "
        "was not found in the Web Map pop-up configuration. No update was made."
    )

In [ ]:
old_survey_id

In [ ]:
map_data_str

## Create a survey

You can create surveys using ArcGIS API for Python with the `create()` method. The `create()` method creates an empty form item and hosted feature service in the folder supplied to the method or a new folder created with the survey. The output of the `create()` method is a single <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.Survey">`Survey`</a> object. 

Surveys created by the `create()` method create the equivalent of a new unpublished web designer survey. After creating a survey with ArcGIS API for Python, if you don't have an XLSForm or existing survey design, you can use the <a href="https://doc.arcgis.com/en/survey123/browser/create-surveys/publishsurvey.htm">Survey123 web designer</a> to design and publish your survey. 

The first step is to connect to your GIS.

Next, a `SurveyManager` is defined, and a new survey is created. A survey in the Survey Manager is a single instance of a survey project that contains the item information and properties and provides access to the underlying survey dataset. For more information on Survey Manager, see the <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.SurveyManager" target="_blank">API Reference for the ArcGIS API for Python</a>.

To create a survey, use the `create()` method on a `SurveyManager` instance. 

The `create()` method only requires the `title` parameter. By default, the `create()` method will automatically create a folder using the `Survey-{title}` naming convention consistent with other Survey123 publishing apps. Alternatively, if you want to store your survey in a folder that already exists in your content, you can use the `folder` parameter and specify the name or folder ID.

The result of the `create()` method is a <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#survey" target="_blank">Survey</a> object. At this stage, the survey is in a draft state and has no schema or questions. You can use the Survey123 web designer to design your survey and publish from there, use the `publish()` method with an existing XLSForm design, or use a third-party Python package to design your survey directly in Python and call the `publish()` method. 

## Publish a survey

To publish a survey using the `publish()` method, an existing `Survey` object is required. The `Survey` can be an unpublished survey created by the `create()` method or an existing published survey in your content. It can also be an unpublished blank survey created with the Survey123 web designer (any designs will be overwritten by the `publish()` method's required XLSForm.). 

When publishing surveys with the `publish()` method, important considerations are as follows:

- Any survey is treated as a survey published with Survey123 Connect, which means you can no longer use the web designer to edit the portions of the survey that are managed by the XLSForm. For example, the survey title and survey questions cannot be edited. Themes, webhooks, and sharing options can still be edited in the web designer. 
- New surveys will create and use a `_form` view as the submission endpoint, which means you can no longer update the feature service schema in Survey123 Connect. 
- Existing surveys will maintain their submission endpoint.

A `schema_changes` parameter is available to modify the schema of your submission endpoint. For more information on the `schema_changes` parameter please see the <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.Survey.publish" target="_blank">ArcGIS API for Python</a> documentation.

In this example, the required `xlsform` parameter is set to the path where the XLSForm is located and optional parameters are set: 

- `info` is set to enable the Inbox.
- `enable_delete_protection` is enabled for the form item and its related content. 

In [ ]:
# Instatiate surveymanager object
survey_manager = arcgis.apps.survey123.SurveyManager(gis)

# Path to XLSForm file
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\Survey123_Volledige_Generatie_20260708.xlsx")

In [ ]:
# 1. Haal de unieke ID van het hoofdformulier op via de naam
survey_name = 'LSVI App Test Auto'
old_survey_id = utils.get_survey_id_by_name(gis, survey_name)

if not old_survey_id:
    print(
        f" [!] Geen survey gevonden met de naam '{survey_name}'. Annuleren."
    )

form_item = gis.content.get(old_survey_id)
if not form_item:
    print(f" [!] Kon het item met ID {old_survey_id} niet ophalen.")

# Sla de unieke formulier-ID op (Survey123 gebruikt deze ID vaak in de servicenamen)
form_id = form_item.id
user = gis.users.get(form_item.owner)

# 2. Zoek de map waarin deze specifieke survey leeft
folder_name, folder_id = utils.browse_folders_for_survey(gis, old_survey_id)
print(f"Survey bevindt zich in map: '{folder_name}' (ID: {folder_id})")

# Haal ALLE items op die in deze map staan
all_folder_items = user.items(folder=folder_id)

# 3. Filter de items: We selecteren alleen items die bij DEZE survey horen
dependent_items_to_delete = []
source_layers_to_delete = []

print("Analyseren van items in de map...")
for item in all_folder_items:
    print(item.id)
    # We identificeren de survey-elementen op basis van:
    # - Exacte match met de Formulier ID (form_id)
    # - De surveynaam die aan het begin van de titel staat (bijv. 'LSVI App Test Auto Map')
    # - Of de form_id die voorkomt in de titel/naam (Survey123 Connect gebruikt: survey123_<id>_fieldworker)
    is_target_item = (
        item.id == form_id
        or item.title.lower().startswith(survey_name.lower())
        or form_id in item.title
        or (item.name and form_id in item.name)
    )

    if is_target_item:
        # Onderscheid maken tussen bron-lagen en afhankelijke lagen (Views/Forms/Maps)
        # Dit is CRUCIAL: Je kunt een bronservice pas deleten als de Views en Web Maps weg zijn!
        is_source_layer = (
            item.type == "Feature Service"
            and "ViewService" not in item.typeKeywords
        )

        if is_source_layer:
            source_layers_to_delete.append(item)
        else:
            dependent_items_to_delete.append(item)

# Toon overzicht van wat er verwijderd gaat worden
print(
    f"\nGevonden afhankelijke onderdelen ({len(dependent_items_to_delete)}): {[i.title for i in dependent_items_to_delete]}"
)
print(
    f"Gevonden hoofd-Feature Services ({len(source_layers_to_delete)}): {[i.title for i in source_layers_to_delete]}\n"
)

if not dependent_items_to_delete and not source_layers_to_delete:
        print(" -> Geen gekoppelde elementen gevonden om te verwijderen.")

# 4. LOOP 1: Verwijder eerst alle afhankelijke zaken (Form, Views, Web Maps)
for item in dependent_items_to_delete:
    try:
        print(
            f" -> Beveiliging opheffen en verwijderen: {item.title} ({item.type})"
        )
        item.protect(enable=False)  # Schakel de verwijder-beveiliging uit
        item.delete()  # Verwijder het item
    except Exception as e:
        print(
            f" [!] Fout bij het verwijderen van afhankelijk item '{item.title}': {e}"
        )

# 5. LOOP 2: Nu de views en kaarten weg zijn, kunnen we de hoofd Feature Service(s) veilig wissen
for item in source_layers_to_delete:
    try:
        print(
            f" -> Beveiliging opheffen en verwijderen hoofdlaag: {item.title} ({item.type})"
        )
        item.protect(enable=False)
        item.delete()
    except Exception as e:
        print(
            f" [!] Fout bij het verwijderen van hoofd Feature Service '{item.title}': {e}"
        )

print(
    f"=== Voltooid: Elementen voor '{survey_name}' zijn succesvol opgeruimd. Map '{folder_name}' is behouden. ===\n"
)

In [ ]:
# We first have to wipe the existing folder and its contents (feature layers, forms, etc.)

# Fetch old survey ID by name before deletion
old_survey_id = utils.get_survey_id_by_name(gis, 'LSVI App Test Auto')
print(f"Bestaande survey ({old_survey_id}) aan het updaten...")

form_item = gis.content.get(old_survey_id)
user = gis.users.get(form_item.owner)
item_title = form_item.title

folder_name, folder_id = utils.browse_folders_for_survey(gis, old_survey_id)
print(folder_name, folder_id)

# Fetch all items residing inside this specific folder
folder_items = user.items(folder=folder_name)

# First remove dependent items: forms, views, maps
for item in folder_items:
    # Check if the item is the absolute root/source Feature Service
    is_source_layer = (item.type == "Feature Service" and "ViewService" not in item.typeKeywords)
    
    if not is_source_layer:
        try:
            print(f"Disabling protection and deleting: {item.title} ({item.type})")
            item.protect(enable=False)  # Lift the protection lock
            item.delete()               # Delete the item
        except Exception as e:
            print(f" [!] Failed to delete dependent item '{item.title}': {e}")

# We query the folder again to grab the leftover source layer(s)
remaining_items = user.items(folder=folder_name)
for item in remaining_items:
    try:
        print(f"Disabling protection and deleting source layer: {item.title} ({item.type})")
        item.protect(enable=False)
        item.delete()
    except Exception as e:
        print(f" [!] Failed to delete source layer '{item.title}': {e}")

# Only delete if it's an actual subfolder and not your root directory
if folder_name and folder_name.lower() != "root":
    try:
        print(f"Removing empty folder structure: '{folder_name}'...")
        gis.content.delete_folder(folder=folder_name, owner=form_item.owner)
        print("Successfully wiped folder and all its contents.")
    except Exception as e:
        print(f" [!] Failed to delete the folder itself: {e}")
else:
    print("Items deleted from root. Skipping folder deletion.")

In [ ]:
# Create new folder for survey
folder_name = "Survey-LSVI App Test Auto"
map_object = gis.content.create_folder(folder=folder_name)
print(f"Succes! De map '{folder_name}' is aangemaakt.")
print(map_object)

# Create new survey in dedicated folder
survey_manager = arcgis.apps.survey123.SurveyManager(gis)
new_survey = survey_manager.create(
    title = "LSVI App Test Auto",
    folder = "Survey-LSVI App Test Auto",
    tags = "LSVI, Survey123, Python",
    summary = "Test Survey123 for LSVI field work.",
    description = "This survey is created for testing the LSVI field work app. We tried to publish this survey automatically from an XLSX form.", 
    thumbnail = r"../inbo_logo.jpg"
)

published_survey = new_survey.publish(
    xlsform=str(xlsform_path), 
    # info=
    #     {"queryInfo": {
    #         "mode": "manual",
    #         "editEnabled": True,
    #         "copyEnabled": False
    #         }
    #     }, 
    enable_delete_protection=False, # We want to be able to delete and overwrite the survey
    enable_sync=True, # Enable sync for offline use,
    thumbnail = r"../inbo_logo.jpg",
    schema_changes=True
    )

# Fetch object ID of new survey
new_survey_id = utils.get_survey_id_by_name(gis, 'LSVI App Test Auto')
print(new_survey_id)

### Sync BWK Field Map Web Map With Latest Form ID

### Attempt to sync our surveys 

In [ ]:
# # Initialiseer de Survey123 Manager
# survey_manager = survey123.SurveyManager(gis)

# # Make nieuwe Web Map, Feature Layer en Form Item aan.
# def publiceer_nieuwe_survey(xlsform_path):
#     print("Nieuwe survey aanmaken...")
#     nieuwe_survey = survey_manager.create(
#         title="LSVI Survey - Automatisch",
#         folder="LSVI App Test",  # Hier specificeer je jouw AGOL map!
#         tags = "Water surveys, Survey123, Python",
#         summary = "lsvi",
#         description = "Survey automatisch gegenereerd via Python script op basis van vereisten en soortenlijsten in LSVI package. Testen van dynamische vraaggeneratie en koppeling met AGOL.", 
#     )
#     print(f"✅ Succes! Nieuwe survey aangemaakt met ID: {nieuwe_survey}")

#     # En publiceren
#     nieuwe_survey.publish(form=xlsform_path,
#                           info=
#                             {"queryInfo": {
#                                 "mode": "manual",
#                                 "editEnabled": True,
#                                 "copyEnabled": False
#                                 }
#                             }, 
#                         enable_delete_protection=False, # We want to be able to delete and overwrite the survey
#                         )
#     print("✅ Survey succesvol gepubliceerd!")
#     return nieuwe_survey

# # Update bestaande survey
# def update_bestaande_survey(form_item_id, xlsform_path):
#     print(f"Bestaande survey ({form_item_id}) aan het updaten...")
    
#     # Haal de bestaande survey op
#     mijn_survey = survey_manager.get(form_item_id)
    
#     # Overschrijf het formulier met het nieuwe Excel-bestand
#     mijn_survey.update(form=xlsform_path)
#     print("✅ Survey succesvol overschreven met de nieuwe vragen!")

In [ ]:
# form_item_id = "9d86f50905c143c28fc4f419ef575a49"  # Vervang dit door het ID van jouw bestaande survey
# xlsform_path = r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\Survey123_Volledige_Generatie_20260601.xlsx"
# print(f"Bestaande survey ({form_item_id}) aan het updaten...")

# # Haal de bestaande survey op
# mijn_survey = survey_manager.get(form_item_id)

# # Overschrijf het formulier met het nieuwe Excel-bestand
# mijn_survey.update(form=xlsform_path)
# print("✅ Survey succesvol overschreven met de nieuwe vragen!")


### Better way of deleted

- dont remove complete folder
- just find objects linked to survey and remove those


def delete_specific_survey(gis, survey_name: str):
    """Verwijdert een specifieke Survey123 survey en alle daaraan gekoppelde

    elementen (Form, Feature Layers, Views, Web Maps) uit ArcGIS Online,
    zonder de rest van de map of andere surveys te beschadigen.
    """
    print(f"=== Start gerichte verwijdering voor survey: '{survey_name}' ===")

    # 1. Haal de unieke ID van het hoofdformulier op via de naam
    old_survey_id = utils.get_survey_id_by_name(gis, survey_name)

    if not old_survey_id:
        print(
            f" [!] Geen survey gevonden met de naam '{survey_name}'. Annuleren."
        )
        return

    form_item = gis.content.get(old_survey_id)
    if not form_item:
        print(f" [!] Kon het item met ID {old_survey_id} niet ophalen.")
        return

    # Sla de unieke formulier-ID op (Survey123 gebruikt deze ID vaak in de servicenamen)
    form_id = form_item.id
    user = gis.users.get(form_item.owner)

    # 2. Zoek de map waarin deze specifieke survey leeft
    folder_name, folder_id = utils.browse_folders_for_survey(gis, old_survey_id)
    print(f"Survey bevindt zich in map: '{folder_name}' (ID: {folder_id})")

    # Haal ALLE items op die in deze map staan
    all_folder_items = user.items(folder=folder_name)

    # 3. Filter de items: We selecteren alleen items die bij DEZE survey horen
    dependent_items_to_delete = []
    source_layers_to_delete = []

    print("Analyseren van items in de map...")
    for item in all_folder_items:
        # We identificeren de survey-elementen op basis van:
        # - Exacte match met de Formulier ID (form_id)
        # - De surveynaam die aan het begin van de titel staat (bijv. 'LSVI App Test Auto Map')
        # - Of de form_id die voorkomt in de titel/naam (Survey123 Connect gebruikt: survey123_<id>_fieldworker)
        is_target_item = (
            item.id == form_id
            or item.title.lower().startswith(survey_name.lower())
            or form_id in item.title
            or (item.name and form_id in item.name)
        )

        if is_target_item:
            # Onderscheid maken tussen bron-lagen en afhankelijke lagen (Views/Forms/Maps)
            # Dit is CRUCIAL: Je kunt een bronservice pas deleten als de Views en Web Maps weg zijn!
            is_source_layer = (
                item.type == "Feature Service"
                and "ViewService" not in item.typeKeywords
            )

            if is_source_layer:
                source_layers_to_delete.append(item)
            else:
                dependent_items_to_delete.append(item)

    # Toon overzicht van wat er verwijderd gaat worden
    print(
        f"\nGevonden afhankelijke onderdelen ({len(dependent_items_to_delete)}): {[i.title for i in dependent_items_to_delete]}"
    )
    print(
        f"Gevonden hoofd-Feature Services ({len(source_layers_to_delete)}): {[i.title for i in source_layers_to_delete]}\n"
    )

    if not dependent_items_to_delete and not source_layers_to_delete:
        print(" -> Geen gekoppelde elementen gevonden om te verwijderen.")
        return

    # 4. LOOP 1: Verwijder eerst alle afhankelijke zaken (Form, Views, Web Maps)
    for item in dependent_items_to_delete:
        try:
            print(
                f" -> Beveiliging opheffen en verwijderen: {item.title} ({item.type})"
            )
            item.protect(enable=False)  # Schakel de verwijder-beveiliging uit
            item.delete()  # Verwijder het item
        except Exception as e:
            print(
                f" [!] Fout bij het verwijderen van afhankelijk item '{item.title}': {e}"
            )

    # 5. LOOP 2: Nu de views en kaarten weg zijn, kunnen we de hoofd Feature Service(s) veilig wissen
    for item in source_layers_to_delete:
        try:
            print(
                f" -> Beveiliging opheffen en verwijderen hoofdlaag: {item.title} ({item.type})"
            )
            item.protect(enable=False)
            item.delete()
        except Exception as e:
            print(
                f" [!] Fout bij het verwijderen van hoofd Feature Service '{item.title}': {e}"
            )

    print(
        f"=== Voltooid: Elementen voor '{survey_name}' zijn succesvol opgeruimd. Map '{folder_name}' is behouden. ===\n"
    )
